In [1]:
import ftlelab as fl
# import ftlelab.configs as cfg
import ftlelab.models as fmods
import torch

import ftlelab.utils as utils
import ftlelab.data as datasets
import ftlelab.training as training

import jax
from tqdm import tqdm

import numpy as np
import jax.numpy as jnp
from ftlelab.ftle.jax_transfer import pytorch_dense_to_jax_params
from ftlelab.ftle.jax_core import ftle_field_batched
# jax.config.update("jax_enable_x64", True)
import ftlelab.ftle as ftle

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


%load_ext autoreload
%autoreload 2

In [22]:
experiment_name = "7-grids-datasets-batched" #"9-grids-datasets-batched-initializations"

# 1. Datasets

In [23]:
total_size = 15000
train_size = int(0.8 * total_size)

In [24]:
batch_size=256

In [25]:
### Circles
X_circ, y_circ = datasets.make_circle_dataset(num_samples=total_size,
                                              noise_std=0.0,
                                              seed=42)

data_circ = datasets.make_dataloaders(datasets.split_dataset(X_circ, y_circ, seed=42), batch_size=batch_size)

### Moons
X_moons, y_moons = datasets.make_moons_dataset(num_samples=total_size,
                                               noise_std=0.0,
                                               seed=42)
data_moons = datasets.make_dataloaders(datasets.split_dataset(X_moons, y_moons, seed=42), batch_size=batch_size)

### Spirals
X_spiral, y_spiral = datasets.make_spiral_dataset(num_samples=total_size,
                                                  noise_std=0.0,
                                                  seed=42)
data_spiral = datasets.make_dataloaders(datasets.split_dataset(X_spiral, y_spiral, seed=42), batch_size=batch_size)

### Xor
X_xor, y_xor = datasets.make_xor_dataset(num_samples=total_size,
                                         noise_std=0.0,
                                         seed=42)
data_xor = datasets.make_dataloaders(datasets.split_dataset(X_xor, y_xor, seed=42), batch_size=batch_size)

In [26]:
dataset_names = ["circle", "moons", "spiral", "xor"]
datasets_dict = {
    "circle": data_circ,
    "moons": data_moons,
    "spiral": data_spiral,
    "xor": data_xor,
}

In [27]:
activations = ["tanh", "relu", "leaky_relu", "gelu", "softplus"]

## 2. Experimentation

In [28]:
ell_list = [2, 4, 8, 12]
n_list = [5, 10, 50, 250]

### 2.1. Tanh

In [29]:
networks_grid_configs = {}

for dataset_name in dataset_names:
    networks_grid_configs[dataset_name] = {}
    for activation in activations:
        networks_grid_configs[dataset_name][activation] = {}
        for n in n_list:
            for ell in ell_list: 
                config_key = f"({n}, {ell})-{dataset_name}-{activation}"
                layer_sizes = [2] + [n] * ell + [1]
                networks_grid_configs[dataset_name][activation][config_key] = {
                    "net_params": {
                        "layer_dims": layer_sizes,
                        "init_method": "xavier" if activation in ("tanh", "sigmoid") else "he",
                        "activation": activation,
                        "output_activation": "tanh",
                    },
                    "train_config":
                    training.TrainConfig(
                        lr=1e-2, 
                        epochs=200, # 2500,
                        batch_size=batch_size, # 256,
                        loss='mse',
                        optimizer='sgd',
                        weight_decay=0.0,
                        save_dir=f'experiment_{experiment_name}/{dataset_name}/{activation}',
                        model_name=f"__batch-{batch_size}__{config_key}",
                        momentum=0.0,
                        print_every=100, #200
                    )
                }

In [ ]:
for dataset_name in dataset_names[1:]:
    for activation in activations:
        for network in tqdm(networks_grid_configs[dataset_name][activation].keys()):
            net = fmods.DenseNet(**networks_grid_configs[dataset_name][activation][network]["net_params"])

            trainer = training.Trainer(net, networks_grid_configs[dataset_name][activation][network]["train_config"])
            trainer.train(datasets_dict[dataset_name]['train'], datasets_dict[dataset_name]['val'])

#### Model Loading

In [30]:
networks_grid = {}
for dataset_name in dataset_names:
    networks_grid[dataset_name] = {}
    for activation in activations:
        networks_grid[dataset_name][activation] = {}
        for network in networks_grid_configs[dataset_name][activation].keys():
            config = networks_grid_configs[dataset_name][activation][network]
            net = fmods.DenseNet(**config["net_params"])
            loaded_net = torch.load(f'experiment_{experiment_name}/{dataset_name}/{activation}/LAST_CHECKPOINT__batch-256__{network}.pt')
            net.load_state_dict(loaded_net['model_state_dict'])
            net.eval()
            networks_grid[dataset_name][activation][network] = net

FileNotFoundError: [Errno 2] No such file or directory: 'experiment_7-grids-datasets-batched/circle/softplus/LAST_CHECKPOINT__batch-256__(5, 2)-circle-softplus.pt'

In [31]:
nx, ny = 200, 200

# circ
grid_circ, (XX, YY) = ftle.make_grid2d(-1.01, 1.01, -1.01, 1.01, nx=nx, ny=ny)
grid_circ_jax = jnp.asarray(grid_circ.numpy())

# moons
grid_moons, (XX_m, YY_m) = ftle.make_grid2d(-2, 2, -1.75, 2.25, nx=nx, ny=ny)
grid_moons_jax = jnp.asarray(grid_moons.numpy())

# spiral 
grid_spiral, (XX_s, YY_s) = ftle.make_grid2d(-2.2, 2.2, -2.2, 2.2, nx=nx, ny=ny)
grid_spiral_jax = jnp.asarray(grid_spiral.numpy())


# xor
grid_xor_jax = grid_circ_jax

In [32]:
grids_jax = {
    "circle": grid_circ_jax,
    "moons": grid_moons_jax,
    "spiral": grid_spiral_jax,
    "xor": grid_xor_jax
}

In [ ]:
networks_grid_ftle = {}
for dataset_name in dataset_names:
    networks_grid_ftle[dataset_name] = {}
    for activation in activations:
        networks_grid_ftle[dataset_name][activation] = {}
        for network in networks_grid[dataset_name][activation].keys():
            net = networks_grid[dataset_name][activation][network]
        

            params_jax = pytorch_dense_to_jax_params(net)      

            lam_np = ftle_field_batched(
                model_type="dense",
                params=params_jax,
                X_np=grids_jax[dataset_name],
                layer_spec= ("hidden_k", net.hidden_depth),
                time_L=net.hidden_depth,
                batch_size=2048,
                activation=activation,
                output_activation="tanh",
                dtype="float32",
                max_steps=30
            )
            lam_np.reshape(YY.shape)

            networks_grid_ftle[dataset_name][activation][f"{network}-ftle_field"] = lam_np

In [29]:
np.save(f"experiment_{experiment_name}/networks_grid_ftle_last-hidden.npy", networks_grid_ftle)

In [37]:
networks_grid_ftle_loaded = np.load(f"experiment_{experiment_name}/networks_grid_ftle_final.npy", allow_pickle=True).item()

# Another Way

In [38]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


def plot_ftle_grid(
    networks_grid_ftle,
    dataset_name,
    activation,
    ell_list=(2, 4, 8, 12),
    n_list=(5, 10, 50, 250),
    network_order=None,
    nx=200,
    ny=200,
    extent=(-1.5, 1.5, -1.5, 1.5),
    cmap="RdBu_r",
    vmin=-3,
    vcenter=0,
    vmax=3,
    multiply_by_L=True,
    figsize=None,
    dpi=100,
    add_index=True,
    index_fontsize=14,
    label_fontsize=16,
    cbar_fontsize=16,
    title=None,
):
    """
    Plot an FTLE grid with:

      rows    -> N values from n_list
      columns -> L values from ell_list

    So if:
        n_list   = [10, 50]
        ell_list = [2, 4, 10, 20, 40]
    then the figure is a 2 x 5 grid.

    Expected dictionary structure:
      networks_grid_ftle[dataset_name][activation][network_key] = net

    Assumptions:
    - each `net` is either:
        * a flat FTLE field of length nx * ny
        * or already a 2D array of shape (ny, nx)
    - if network_order is provided, it must be in row-major order with:
        row-major over n_list first, then ell_list within each row:
        [
          (N=n_list[0], L=ell_list[0]),
          (N=n_list[0], L=ell_list[1]),
          ...
          (N=n_list[0], L=ell_list[-1]),
          (N=n_list[1], L=ell_list[0]),
          ...
        ]
    """

    data_dict = networks_grid_ftle[dataset_name][activation]

    n_rows = len(n_list)
    n_cols = len(ell_list)
    n_expected = n_rows * n_cols

    if network_order is None:
        network_keys = list(data_dict.keys())
        if len(network_keys) != n_expected:
            raise ValueError(
                "network_order was not provided, but the number of networks "
                f"({len(network_keys)}) does not match len(n_list)*len(ell_list) = {n_expected}."
            )
    else:
        network_keys = list(network_order)
        if len(network_keys) != n_expected:
            raise ValueError(
                f"network_order must have exactly {n_expected} entries, got {len(network_keys)}."
            )

    if figsize is None:
        figsize = (3.5 * n_cols, 3.5 * n_rows)

    fig, axes = plt.subplots(
        nrows=n_rows,
        ncols=n_cols,
        figsize=figsize,
        dpi=dpi,
        constrained_layout=True,
        squeeze=False,   # crucial for 1x1, 1xN, Nx1, etc.
    )

    norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
    im = None

    for i, net_key in enumerate(network_keys):
        r = i // n_cols   # row -> N
        c = i % n_cols    # col -> L

        ax = axes[r, c]
        result = data_dict[net_key]

        arr = np.asarray(result)
        if arr.ndim == 1:
            field = arr.reshape(ny, nx)
        elif arr.ndim == 2:
            field = arr
        else:
            raise ValueError(
                f"Network '{net_key}' has unsupported shape {arr.shape}. "
                "Expected flat array or 2D field."
            )

        L = ell_list[c]
        N = n_list[r]

        plot_field = L * field if multiply_by_L else field

        im = ax.imshow(
            plot_field,
            extent=extent,
            origin="lower",
            cmap=cmap,
            alpha=0.9,
            norm=norm,
            aspect="auto",
        )

        if add_index:
            x0, x1, y0, y1 = extent
            dx = x1 - x0
            dy = y1 - y0
            ax.text(
                x0 - 0.18 * dx,
                y1 - 0.02 * dy,
                f"[{i + 1}]",
                color="black",
                fontsize=index_fontsize,
                ha="left",
                va="top",
            )

        # top labels = L (columns)
        if r == 0:
            ax.set_xlabel(f"L = {L}", fontsize=label_fontsize)
            ax.xaxis.set_label_position("top")

        # left labels = N (rows)
        if c == 0:
            ax.set_ylabel(f"N = {N}", fontsize=label_fontsize)

        ax.set_xticks([])
        ax.set_yticks([])

    cbar = fig.colorbar(
        im,
        ax=axes.ravel().tolist(),
        orientation="horizontal",
        location="bottom",
        shrink=0.8,
    )

    cbar.set_label(
        r"$L\lambda^{(L)}(\mathbf{x})$" if multiply_by_L else r"$\lambda^{(L)}(\mathbf{x})$",
        fontsize=cbar_fontsize,
    )
    cbar.ax.tick_params(labelsize=max(cbar_fontsize - 2, 8))

    if title is not None:
        fig.suptitle(title, fontsize=label_fontsize + 2)

    plt.savefig('ftle_grid-'+ dataset_name + '_' + activation.replace('_', '-') + '.png', bbox_inches='tight')
    # plt.show()

In [ ]:
for dataset_name in dataset_names:
    for activation in [act for act in activations if act != "softplus"]:
        plot_ftle_grid(
            networks_grid_ftle_loaded,
            dataset_name=dataset_name,
            activation=activation,
            ell_list=[2, 4, 8, 12],
            n_list=[5, 10, 50, 250],
            nx=200,
            ny=200,
            vmin=-4,
            vmax=4,
            multiply_by_L=True
        )


# Plotting Decision level sets

In [13]:
def plot_decision_grid_NL(
    networks_grid,
    dataset_name: str,
    activation: str,
    XX, YY,
    X_val, y_val,
    n_list=(5, 10, 50, 250),
    ell_list=(2, 4, 8, 12),
    network_name_template="N{N}_L{L}",
    device="cpu",
    levels=20,
    cmap="RdBu_r",
    vmin=-1.5,
    vcenter=0.0,
    vmax=1.5,
    figsize=(18, 18),
    dpi=100,
    add_index=True,
    title=None,
):
    """
    Plot decision level sets in a grid:
      rows    = depth    L  (ell_list, top label)
      columns = width    N  (n_list,   left label)

    Row/column convention mirrors plot_ftle_grid_4x4:
      ax = axes[row, col]  where row indexes ell_list and col indexes n_list.

    Parameters
    ----------
    networks_grid         : dict  [dataset_name][activation][model_name] -> nn.Module
    XX, YY                : np.ndarray  meshgrid, shape (ny, nx)
    X_val, y_val          : torch.Tensor  validation set, y in {-1, +1}
    n_list                : sequence of int  hidden-layer widths  -> columns
    ell_list              : sequence of int  number of hidden layers -> rows
    network_name_template : str  f-string with {N} and {L}
                            e.g. "N{N}_L{L}" produces "N50_L4"
    add_index             : bool  add [k] counter in top-left corner
    """
    from matplotlib.colors import TwoSlopeNorm

    nrows = len(n_list)
    ncols = len(ell_list)

    grid = torch.tensor(
        np.column_stack([XX.ravel(), YY.ravel()]),
        dtype=torch.float32,
        device=device,
    )

    X_val_d    = X_val.to(device)
    y_val_flat = y_val.view(-1).to(device)

    activation_dict = networks_grid[dataset_name][activation]
    norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=figsize,
        dpi=dpi,
        constrained_layout=True,
    )
    axes = np.atleast_2d(axes)

    im      = None
    counter = 1

    for r, N in enumerate(n_list):        # rows = N
        for c, L in enumerate(ell_list):  # cols = L
            ax = axes[r, c]

            model_name = network_name_template.format(N=N, L=L)

            if model_name not in activation_dict:
                ax.text(
                    0.5, 0.5, f"Missing:\n{model_name}",
                    ha="center", va="center", fontsize=10,
                    transform=ax.transAxes,
                )
                ax.set_xticks([]); ax.set_yticks([])
                if add_index:
                    ax.text(
                        0.03, 0.94, f"[{counter}]",
                        transform=ax.transAxes,
                        color="black", fontsize=18, va="top",
                    )
                counter += 1
                continue

            model = activation_dict[model_name]
            model.eval()

            with torch.no_grad():
                preds_grid = (
                    model(grid)
                    .squeeze(-1)
                    .cpu()
                    .numpy()
                    .reshape(XX.shape)
                )

            im = ax.contourf(
                XX, YY, preds_grid,
                levels=levels,
                cmap=cmap,
                norm=norm,
            )
            ax.contour(
                XX, YY, preds_grid,
                levels=[0.0],
                colors="k",
                linewidths=0.9,
                linestyles="--",
            )

            with torch.no_grad():
                out_val  = model(X_val_d).squeeze(-1)
                pred_val = torch.where(
                    out_val >= 0.0,
                    torch.ones_like(out_val),
                    -torch.ones_like(out_val),
                )
                acc = (pred_val == y_val_flat.float()).float().mean().item()

            ax.text(
                0.03, 0.04,
                f"acc[test]={acc:.2%}",
                transform=ax.transAxes,
                fontsize=15,
                color="white",
                bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.45, lw=0),
            )

            if add_index:
                ax.text(
                    0.03, 0.94, f"[{counter}]",
                    transform=ax.transAxes,
                    color="black", fontsize=18, va="top",
                )

            ax.set_xticks([]); ax.set_yticks([])

            if r == 0:
                ax.set_title(f"L = {L}", fontsize=20, pad=12)   # top:  L
            if c == 0:
                ax.set_ylabel(f"N = {N}", fontsize=20,
                              rotation=90, labelpad=16)          # left: N

            counter += 1

    if im is not None:
        cbar = fig.colorbar(
            im,
            ax=axes.ravel().tolist(),
            orientation="horizontal",
            location="bottom",
            shrink=0.85,
            pad=0.04,
        )
        cbar.set_label("Network's Decision Level Sets", fontsize=20)
        cbar.ax.tick_params(labelsize=18)

    fig.suptitle(
        title or f"Decision level sets — {dataset_name} / {activation}",
        fontsize=20, y=1.02,
    )

    plt.savefig('decision_contour-'+ dataset_name + '_' + activation.replace('_', '-') + '.png', bbox_inches='tight')
    # plt.show()

In [ ]:
for dataset_name in dataset_names:
    for activation in activations:
        plot_decision_grid_NL(
            networks_grid,
            dataset_name=dataset_name,
            activation=activation,
            XX=XX if dataset_name in ("circle", "xor") else XX_s if dataset_name == "spiral" else XX_m,
            YY=YY if dataset_name in ("circle", "xor") else YY_s if dataset_name == "spiral" else YY_m,
            X_val=next(iter(datasets_dict[dataset_name]["test"]))[:][0],
            y_val=next(iter(datasets_dict[dataset_name]["test"]))[:][1],
            n_list=[5, 10, 50, 250],
            ell_list=[2, 4, 8, 12],
            network_name_template="({N}, {L})-" + f"{dataset_name}-{activation}",
            device="cpu",
            levels=20,
            cmap="RdBu_r",
            vmin=-1.1,
            vcenter=0.0,
            vmax=1.1,
            figsize=(18, 18),
            dpi=100,
            add_index=True,
        )

#### Plotting

In [ ]:
# def infer_ftle_limits(ftle_history, symmetric=True, q=0.995):
#     layer_keys = ["layer-1", "layer-2", "layer-3", "layer-4", "layer-5", "output"]
#     vals = []
#     for key in layer_keys:
#         for arr in ftle_history[key]:
#             vals.append(np.ravel(np.asarray(arr)))
#     vals = np.concatenate(vals)

#     if symmetric:
#         m = np.quantile(np.abs(vals), q)
#         return -float(m), 0.0, float(m)
#     else:
#         lo = float(np.quantile(vals, 1 - q))
#         hi = float(np.quantile(vals, q))
#         return lo, 0.0, hi